# Multimodal — essais de pipeline CPDP

Essais inter-projets antérieurs au script retenu : classes `HybridBugPredictorV1/V2`, CodeBERT pré-entraîné en 768 dimensions, 20 époques. L'exécution conservée s'arrête au début de l'entraînement sur Camel. Le script CPDP retenu est `scripts/multimodal_cpdp_hybride_fullpretrained.py`.

**Statut :** Exploratoire

*Mémoire de maîtrise — Mireille Kafando, UQTR · 2025*

> Notebook de recherche conçu pour Google Colab. Les chemins `/content/drive/MyDrive/bug/...` sont à adapter (voir `data/README.md`). Les sorties ont été retirées.


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 25
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 384
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'

    # Paramètres pour l'équilibrage
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé
class BugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)

        # Vérifier la cohérence des tailles
        assert len(self.metrics) == len(self.labels), "Mismatch entre metrics et labels"
        assert len(self.code_tokens['input_ids']) == len(self.labels), "Mismatch entre code_tokens et labels"

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx]

# STRATÉGIE 1: Modèle hybride (encoder from scratch + decoder pré-entraîné)
class HybridTransBugModel(nn.Module):
    def __init__(self, input_metric_dim=None):
        super().__init__()
        # Projection des métriques vers la dimension d'embedding
        self.metric_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM) if input_metric_dim is None else nn.Linear(input_metric_dim, Config.EMBEDDING_DIM),
            nn.BatchNorm1d(Config.EMBEDDING_DIM),
            nn.Dropout(Config.DROPOUT)
        )

        # Encoder transformer FROM SCRATCH pour les métriques
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=Config.EMBEDDING_DIM,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            activation='ReLu',
            batch_first=True,
            norm_first=True
        )
        self.metrics_encoder = nn.TransformerEncoder(encoder_layer, num_layers=Config.NUM_LAYERS)

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        roberta_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(roberta_config)

        # Projection pour aligner les dimensions RoBERTa -> Config.EMBEDDING_DIM
        self.code_projection = nn.Linear(roberta_config.hidden_size, Config.EMBEDDING_DIM)

        # Couche d'attention pour combiner les représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Traitement des métriques avec encoder from scratch
        metrics_proj = self.metric_projection(metrics)
        metrics_proj = metrics_proj.unsqueeze(1)  # [B, 1, 384]
        metrics_encoded = self.metrics_encoder(metrics_proj)  # [B, 1, 384]

        # Traitement du code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_embed = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_embed = self.code_projection(code_embed)  # [B, seq_len, 384]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query
            code_embed,      # Key
            code_embed       # Value
        )

        # Fusion avec skip connection
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# STRATÉGIE 2: Modèle entièrement pré-entraîné (encoder et decoder pré-entraînés)
class FullPretrainedModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder PRÉ-ENTRAÎNÉ pour les métriques (adaptation de RoBERTa)
        self.metrics_encoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.metrics_encoder_config.hidden_size = Config.EMBEDDING_DIM
        self.metrics_encoder = RobertaModel(self.metrics_encoder_config)

        # Projection pour les métriques numériques vers le format attendu par transformer
        self.metrics_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.Dropout(Config.DROPOUT)
        )

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        self.code_decoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(self.code_decoder_config)

        # Projection pour aligner les dimensions
        self.code_projection = nn.Linear(self.code_decoder_config.hidden_size, Config.EMBEDDING_DIM)

        # Couche d'attention croisée pour combiner les deux représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Projeter les métriques et les adapter au format attendu
        metrics_embedded = self.metrics_projection(metrics).unsqueeze(1)  # [B, 1, 384]

        # Créer un masque factice pour les métriques (toutes valides)
        metrics_attention_mask = torch.ones(metrics_embedded.size(0), 1).to(metrics.device)

        # Encoder les métriques avec un transformer pré-entraîné
        metrics_encoder_output = self.metrics_encoder(
            inputs_embeds=metrics_embedded,
            attention_mask=metrics_attention_mask
        )
        metrics_encoded = metrics_encoder_output.last_hidden_state  # [B, 1, 384]

        # Encoder le code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_encoded = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_encoded = self.code_projection(code_encoded)  # [B, seq_len, 384]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query
            code_encoded,     # Key
            code_encoded      # Value
        )

        # Fusion avec skip connection
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# Fonction d'équilibrage avec BorderlineSMOTE
def apply_borderline_smote(X, y, code_lines):
    """
    Applique BorderlineSMOTE pour équilibrer les classes
    """
    print(f"Distribution avant équilibrage: {Counter(y)}")

    # Vérifier qu'il y a assez d'échantillons minoritaires
    min_samples = min(Counter(y).values())
    k_neighbors = min(3, min_samples - 1) if min_samples > 1 else 1

    try:
        smote = BorderlineSMOTE(random_state=42, k_neighbors=k_neighbors, kind='borderline-1')
        X_resampled, y_resampled = smote.fit_resample(X, y)
        print("✓ BorderlineSMOTE appliqué avec succès")
    except Exception as e:
        print(f"Erreur BorderlineSMOTE: {e}, utilisation de SMOTE standard")
        smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
        X_resampled, y_resampled = smote.fit_resample(X, y)

    print(f"Distribution après équilibrage: {Counter(y_resampled)}")

    # Adapter la liste de code
    n_original = len(code_lines)
    n_resampled = len(X_resampled)

    if n_resampled > n_original:
        # Mapping pour les indices répliqués
        indices_map = []
        for i in range(n_resampled):
            if i < n_original:
                indices_map.append(i)
            else:
                # Pour les nouveaux échantillons, choisir aléatoirement
                indices_map.append(np.random.randint(0, n_original))

        code_lines_resampled = [code_lines[idx] for idx in indices_map]
    else:
        code_lines_resampled = code_lines[:n_resampled]

    return X_resampled, y_resampled, code_lines_resampled

# Calcul des poids de classe
def calculate_class_weights(labels):
    """Calcule les poids de classe pour équilibrer la loss"""
    class_counts = Counter(labels)
    total_samples = len(labels)
    n_classes = len(class_counts)

    weights = {}
    for class_id, count in class_counts.items():
        weights[class_id] = total_samples / (n_classes * count)

    # Convertir en tensor PyTorch
    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])
    return weight_tensor

# Prétraitement des métriques
def preprocess_metrics(df):
    """Préprocessing robuste des métriques"""
    if 'name' in df.columns:
        df = df.drop(columns=['name'])

    df['bug'] = (df['bug'] > 0).astype(int)
    features = df.drop(columns=['bug'])
    labels = df['bug'].values

    # Traiter les valeurs manquantes
    if features.isnull().any().any():
        print("Valeurs manquantes détectées, remplacement par la médiane")
        features = features.fillna(features.median())

    # Supprimer les features avec variance nulle
    var_threshold = 0.01
    low_var_cols = features.var() < var_threshold
    if low_var_cols.any():
        print(f"Suppression de {low_var_cols.sum()} colonnes avec faible variance")
        features = features.loc[:, ~low_var_cols]

    # Normalisation
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    print(f"Distribution des classes: {Counter(labels)}")

    return features_scaled, labels, scaler

# Chargement des métriques et du code
def load_code_and_metrics(metrics_path, code_path):
    """Charge les métriques et le code source"""
    df_metrics = pd.read_csv(metrics_path)
    code_lines = []

    if os.path.exists(code_path):
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        with open(os.path.join(root, file), 'r', encoding='utf-8', errors='ignore') as f:
                            code = f.read()
                            code = code.strip()
                            if len(code) > 0:
                                code_lines.append(code)
                    except Exception as e:
                        print(f"Erreur lecture fichier {file}: {e}")
                        continue

    # S'assurer qu'on a au moins autant de fichiers code que de métriques
    if len(code_lines) < len(df_metrics):
        print(f"Attention: {len(code_lines)} fichiers code vs {len(df_metrics)} métriques")
        while len(code_lines) < len(df_metrics):
            if len(code_lines) > 0:
                code_lines.append(code_lines[len(code_lines) % len(code_lines)])
            else:
                code_lines.append("// Empty code file")

    return df_metrics, code_lines

# Préparation du dataset avec BorderlineSMOTE
def prepare_single_dataset(metrics_path, code_path, tokenizer, apply_balancing=True):
    """Prépare un dataset - VERSION CORRIGÉE"""
    df_metrics, code_lines = load_code_and_metrics(metrics_path, code_path)
    features, labels, scaler = preprocess_metrics(df_metrics)

    # Assurer la cohérence des tailles
    min_len = min(len(features), len(code_lines), len(labels))
    features = features[:min_len]
    code_lines = code_lines[:min_len]
    labels = labels[:min_len]

    # Appliquer l'équilibrage seulement si demandé
    if apply_balancing and len(set(labels)) > 1:
        try:
            features, labels, code_lines = apply_borderline_smote(features, labels, code_lines)
        except Exception as e:
            print(f"Erreur lors de l'équilibrage: {e}")
            print("Utilisation des données originales")

    # Tokenisation du code
    try:
        code_tokens = tokenizer(
            code_lines,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512
        )
    except Exception as e:
        print(f"Erreur lors de la tokenisation: {e}")
        code_tokens = {
            'input_ids': torch.zeros((len(features), 512), dtype=torch.long),
            'attention_mask': torch.zeros((len(features), 512), dtype=torch.long)
        }

    return features, code_tokens, labels, scaler

# Préparation de datasets multiples
def prepare_multiple_datasets(metrics_paths, code_paths, tokenizer, apply_balancing=True):
    """Prépare et combine plusieurs datasets - VERSION CORRIGÉE"""
    all_features, all_labels, all_code_lines = [], [], []

    # ÉTAPE 1: Charger et combiner TOUTES les données SANS équilibrage
    for metrics_path, code_path in zip(metrics_paths, code_paths):
        print(f"Chargement de {metrics_path}...")

        # Charger sans équilibrage individuel
        df_metrics, code_lines = load_code_and_metrics(metrics_path, code_path)
        features, labels, scaler = preprocess_metrics(df_metrics)

        # Assurer la cohérence des tailles
        min_len = min(len(features), len(code_lines), len(labels))
        features = features[:min_len]
        code_lines = code_lines[:min_len]
        labels = labels[:min_len]

        print(f"  Dataset {metrics_path}: {Counter(labels)}")

        all_features.append(features)
        all_labels.extend(labels)
        all_code_lines.extend(code_lines)

    # ÉTAPE 2: Concaténer toutes les données
    X_combined = np.vstack(all_features)
    y_combined = np.array(all_labels)
    code_combined = all_code_lines

    print(f"\nDonnées combinées - Distribution avant équilibrage: {Counter(y_combined)}")

    # ÉTAPE 3: Appliquer BorderlineSMOTE UNE SEULE FOIS sur l'ensemble combiné
    if apply_balancing and len(set(y_combined)) > 1:
        try:
            X_balanced, y_balanced, code_balanced = apply_borderline_smote(
                X_combined, y_combined, code_combined
            )
            print(f"Données finales - Distribution après équilibrage: {Counter(y_balanced)}")
        except Exception as e:
            print(f"Erreur lors de l'équilibrage: {e}")
            print("Utilisation des données originales")
            X_balanced, y_balanced, code_balanced = X_combined, y_combined, code_combined
    else:
        X_balanced, y_balanced, code_balanced = X_combined, y_combined, code_combined

    # ÉTAPE 4: Tokeniser le code source équilibré
    try:
        print("Tokenisation du code source...")
        combined_code_tokens = tokenizer(
            code_balanced,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512
        )
        print(f"✓ Tokenisation réussie: {len(code_balanced)} échantillons")
    except Exception as e:
        print(f"Erreur lors de la tokenisation: {e}")
        combined_code_tokens = {
            'input_ids': torch.zeros((len(X_balanced), 512), dtype=torch.long),
            'attention_mask': torch.zeros((len(X_balanced), 512), dtype=torch.long)
        }

    return X_balanced, combined_code_tokens, y_balanced

# Évaluation
def evaluate(model, dataloader, criterion=None):
    """Évaluation robuste du modèle"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for metrics, code_tokens, labels in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculer les métriques
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    try:
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    except:
        f1 = 0.0

    accuracy = accuracy_score(all_labels, all_preds)

    # Calcul robuste de sensibilité et spécificité
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)

        if sens > 0 and spec > 0:
            g_mean = gmean([sens, spec])
        else:
            g_mean = 0.0
    else:
        sens = spec = g_mean = 0.0

    avg_loss = total_loss / len(all_labels) if criterion is not None and len(all_labels) > 0 else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# Fonction d'entraînement principale
def train_model(model_type="hybrid"):
    """Version corrigée de train_model"""
    print(f"\n{'='*50}")
    print(f"ENTRAÎNEMENT: {model_type.upper()}")
    print(f"{'='*50}")

    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Définir les chemins
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/xerces/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/xerces/"

    # Versions pour l'entraînement
    train_versions = ["1.1", "1.2", "1.3"]
    metrics_train_paths = [f"{base_metrics_path}xerces-{v}.csv" for v in train_versions]
    code_train_paths = [f"{base_code_path}xerces-{v}" for v in train_versions]

    # Version pour la validation
    val_version = "1.4.4"
    metrics_val_path = f"{base_metrics_path}xerces-{val_version}.csv"
    code_val_path = f"{base_code_path}xerces-{val_version}"

    # ✅ CORRECTION: Préparer les datasets d'entraînement avec BorderlineSMOTE appliqué UNE FOIS
    print("Préparation des données d'entraînement avec BorderlineSMOTE...")
    print("Chargement de toutes les versions d'entraînement...")
    X_train, train_code_tokens, y_train = prepare_multiple_datasets(
        metrics_train_paths, code_train_paths, tokenizer, apply_balancing=True
    )

    # Préparer le dataset de validation (sans équilibrage)
    print("Préparation des données de validation...")
    X_val, code_val_tokens, y_val, _ = prepare_single_dataset(
        metrics_val_path, code_val_path, tokenizer, apply_balancing=False
    )

    # Créer les datasets
    train_ds = BugDataset(X_train, train_code_tokens, y_train)
    val_ds = BugDataset(X_val, code_val_tokens, y_val)

    # Créer les DataLoaders
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Calculer les poids de classe
    class_weights = calculate_class_weights(y_train).to(Config.DEVICE)
    print(f"Poids des classes: {class_weights}")

    # Initialiser le modèle selon le type
    if model_type == "full_pretrained":
        model = FullPretrainedModel().to(Config.DEVICE)
        print("✓ Modèle avec ENCODER et DECODER pré-entraînés")
    else:
        model = HybridTransBugModel().to(Config.DEVICE)
        print("✓ Modèle avec ENCODER from scratch et DECODER pré-entraîné")

    # Optimiseur avec learning rates différenciés
    if model_type == "full_pretrained":
        # Pour le modèle entièrement pré-entraîné, utiliser des LR plus faibles pour les parties pré-entraînées
        optimizer = torch.optim.AdamW([
            {'params': model.metrics_encoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters()
                       if 'metrics_encoder' not in n and 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)
    else:
        # Pour le modèle hybride, seul le decoder est pré-entraîné
        optimizer = torch.optim.AdamW([
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters() if 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)

    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=True
    )

    # Loss avec pondération et label smoothing
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=Config.LABEL_SMOOTHING)

    # Listes pour le monitoring
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    f1_scores, gmean_scores = [], []

    best_f1 = 0
    patience_counter = 0

    for epoch in range(Config.EPOCHS):
        # Entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for batch_idx, (metrics, code_tokens, labels) in enumerate(train_loader):
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)

            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Métriques d'entraînement
        train_loss /= len(train_preds)
        train_acc = accuracy_score(train_labels_list, train_preds)
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        # Scheduler step
        scheduler.step(f1)

        print(f"Epoch {epoch+1}/{Config.EPOCHS}:")
        print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
        print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")
        print(f"  Val: Sensitivité={sens:.4f}, Spécificité={spec:.4f}")

        # Sauvegarde du meilleur modèle
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}best_model_{model_type}.pt")
            joblib.dump(cm, f"{Config.SAVE_PATH}conf_matrix_{model_type}.pkl")
            joblib.dump(val_preds, f"{Config.SAVE_PATH}val_preds_{model_type}.pkl")
            joblib.dump(val_labels, f"{Config.SAVE_PATH}val_labels_{model_type}.pkl")
            print(f"  *** Nouveau meilleur F1: {f1:.4f} ***")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= 20:
            print("Early stopping déclenché")
            break

    print(f"\n{model_type.upper()} - Meilleur F1 score: {best_f1:.4f}")

    # Visualisations
    plt.figure(figsize=(15, 5))

    # Courbe 1: Loss
    plt.subplot(1, 3, 1)
    epochs_range = range(1, len(train_losses) + 1)
    plt.plot(epochs_range, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs_range, val_losses, 'r-', label='Val Loss')
    plt.title(f'{model_type.capitalize()} - Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Courbe 2: Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, train_accuracies, 'b-', label='Train Accuracy')
    plt.plot(epochs_range, val_accuracies, 'r-', label='Val Accuracy')
    plt.title(f'{model_type.capitalize()} - Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Courbe 3: F1-score et G-mean
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, f1_scores, 'g-', label='F1-Score')
    plt.plot(epochs_range, gmean_scores, 'm-', label='G-Mean')
    plt.title(f'{model_type.capitalize()} - F1-Score and G-Mean')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"{Config.SAVE_PATH}training_curves_{model_type}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Matrice de confusion finale
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Bug', 'Bug'],
                yticklabels=['Non-Bug', 'Bug'])
    plt.title(f'{model_type.capitalize()} - Confusion Matrix (Best Model)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f"{Config.SAVE_PATH}confusion_matrix_{model_type}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Sauvegarder les métriques
    results = {
        'model_type': model_type,
        'best_f1': best_f1,
        'final_accuracy': val_accuracies[-1] if val_accuracies else 0,
        'final_sensitivity': sens,
        'final_specificity': spec,
        'final_gmean': gmean,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'f1_scores': f1_scores,
        'gmean_scores': gmean_scores
    }

    joblib.dump(results, f"{Config.SAVE_PATH}training_results_{model_type}.pkl")

    return model, results

def compare_models():
    """Compare les deux stratégies de modèles"""
    print("\n" + "="*60)
    print("COMPARAISON DES DEUX STRATÉGIES")
    print("="*60)

    # Entraîner le modèle hybride
    print("\n1. STRATÉGIE HYBRIDE (Encoder from scratch + Decoder pré-entraîné)")
    model_hybrid, results_hybrid = train_model("hybrid")

    # Entraîner le modèle entièrement pré-entraîné
    print("\n2. STRATÉGIE ENTIÈREMENT PRÉ-ENTRAÎNÉE (Encoder et Decoder pré-entraînés)")
    model_full, results_full = train_model("full_pretrained")

    # Comparaison des résultats
    print("\n" + "="*60)
    print("RÉSULTATS COMPARATIFS")
    print("="*60)

    print(f"{'Métrique':<20} {'Hybride':<15} {'Full Pretrained':<15} {'Meilleur':<15}")
    print("-" * 70)

    metrics_comparison = [
        ('F1-Score', results_hybrid['best_f1'], results_full['best_f1']),
        ('Accuracy', results_hybrid['final_accuracy'], results_full['final_accuracy']),
        ('Sensitivity', results_hybrid['final_sensitivity'], results_full['final_sensitivity']),
        ('Specificity', results_hybrid['final_specificity'], results_full['final_specificity']),
        ('G-Mean', results_hybrid['final_gmean'], results_full['final_gmean'])
    ]

    for metric_name, hybrid_score, full_score in metrics_comparison:
        better = "Hybride" if hybrid_score > full_score else "Full Pretrained" if full_score > hybrid_score else "Égalité"
        print(f"{metric_name:<20} {hybrid_score:<15.4f} {full_score:<15.4f} {better:<15}")

    # Graphique comparatif
    plt.figure(figsize=(12, 8))

    # Comparaison F1-Score au fil des époques
    plt.subplot(2, 2, 1)
    epochs_hybrid = range(1, len(results_hybrid['f1_scores']) + 1)
    epochs_full = range(1, len(results_full['f1_scores']) + 1)
    plt.plot(epochs_hybrid, results_hybrid['f1_scores'], 'b-', label='Hybride', linewidth=2)
    plt.plot(epochs_full, results_full['f1_scores'], 'r-', label='Full Pretrained', linewidth=2)
    plt.title('Comparaison F1-Score')
    plt.xlabel('Epochs')
    plt.ylabel('F1-Score')
    plt.legend()
    plt.grid(True)

    # Comparaison G-Mean
    plt.subplot(2, 2, 2)
    plt.plot(epochs_hybrid, results_hybrid['gmean_scores'], 'b-', label='Hybride', linewidth=2)
    plt.plot(epochs_full, results_full['gmean_scores'], 'r-', label='Full Pretrained', linewidth=2)
    plt.title('Comparaison G-Mean')
    plt.xlabel('Epochs')
    plt.ylabel('G-Mean')
    plt.legend()
    plt.grid(True)

    # Comparaison Accuracy
    plt.subplot(2, 2, 3)
    plt.plot(epochs_hybrid, results_hybrid['val_accuracies'], 'b-', label='Hybride', linewidth=2)
    plt.plot(epochs_full, results_full['val_accuracies'], 'r-', label='Full Pretrained', linewidth=2)
    plt.title('Comparaison Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Graphique en barres des métriques finales
    plt.subplot(2, 2, 4)
    metrics_names = ['F1-Score', 'Accuracy', 'Sensitivity', 'Specificity', 'G-Mean']
    hybrid_values = [results_hybrid['best_f1'], results_hybrid['final_accuracy'],
                    results_hybrid['final_sensitivity'], results_hybrid['final_specificity'],
                    results_hybrid['final_gmean']]
    full_values = [results_full['best_f1'], results_full['final_accuracy'],
                  results_full['final_sensitivity'], results_full['final_specificity'],
                  results_full['final_gmean']]

    x = np.arange(len(metrics_names))
    width = 0.35

    plt.bar(x - width/2, hybrid_values, width, label='Hybride', alpha=0.8)
    plt.bar(x + width/2, full_values, width, label='Full Pretrained', alpha=0.8)

    plt.title('Comparaison des Métriques Finales')
    plt.xlabel('Métriques')
    plt.ylabel('Score')
    plt.xticks(x, metrics_names, rotation=45)
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{Config.SAVE_PATH}models_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Recommandation
    print("\n" + "="*60)
    print("RECOMMANDATION")
    print("="*60)

    if results_hybrid['best_f1'] > results_full['best_f1']:
        best_model = "Hybride"
        best_results = results_hybrid
    else:
        best_model = "Full Pretrained"
        best_results = results_full

    print(f"✓ Le modèle {best_model} obtient les meilleurs résultats")
    print(f"  - F1-Score: {best_results['best_f1']:.4f}")
    print(f"  - G-Mean: {best_results['final_gmean']:.4f}")
    print(f"  - Accuracy: {best_results['final_accuracy']:.4f}")

    # Analyse des avantages
    print(f"\nAnalyse des stratégies:")
    print("• Modèle Hybride:")
    print("  - Encoder from scratch pour les métriques")
    print("  - Decoder pré-entraîné pour le code source")
    print("  - Plus flexible pour l'adaptation aux métriques spécifiques")

    print("• Modèle Full Pretrained:")
    print("  - Encoder et Decoder pré-entraînés")
    print("  - Tire parti de plus de connaissances pré-entraînées")
    print("  - Peut nécessiter moins d'entraînement")

    return best_model, best_results

def main():
    """Fonction principale pour lancer la comparaison"""
    print("🚀 Démarrage de l'entraînement et comparaison des modèles de détection de bugs")
    print("Utilisation de BorderlineSMOTE pour l'équilibrage des classes")

    try:
        best_model, best_results = compare_models()

        print(f"\n🎉 Entraînement terminé avec succès!")
        print(f"Meilleur modèle: {best_model}")
        print(f"Résultats sauvegardés dans: {Config.SAVE_PATH}")

    except Exception as e:
        print(f"❌ Erreur lors de l'entraînement: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 25
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-7
    EMBEDDING_DIM = 384
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'

    # Paramètres pour l'équilibrage
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé
class BugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)

        # Vérifier la cohérence des tailles
        assert len(self.metrics) == len(self.labels), "Mismatch entre metrics et labels"
        assert len(self.code_tokens['input_ids']) == len(self.labels), "Mismatch entre code_tokens et labels"

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx]

# STRATÉGIE 1: Modèle hybride (encoder from scratch + decoder pré-entraîné)
class HybridTransBugModel(nn.Module):
    def __init__(self, input_metric_dim=None):
        super().__init__()
        # Projection des métriques vers la dimension d'embedding
        self.metric_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM, bias = False) if input_metric_dim is None else nn.Linear(input_metric_dim, Config.EMBEDDING_DIM, bias = False),
            nn.BatchNorm1d(Config.EMBEDDING_DIM),
            nn.Dropout(Config.DROPOUT)
        )

        # Encoder transformer FROM SCRATCH pour les métriques
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=Config.EMBEDDING_DIM,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            activation='relu'

        )
        self.metrics_encoder = nn.TransformerEncoder(encoder_layer, num_layers=Config.NUM_LAYERS)

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        roberta_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(roberta_config)

        # Projection pour aligner les dimensions RoBERTa -> Config.EMBEDDING_DIM
        self.code_projection = nn.Linear(roberta_config.hidden_size, Config.EMBEDDING_DIM)

        # Couche d'attention pour combiner les représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,

        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.ReLU()

        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Traitement des métriques avec encoder from scratch
        metrics_proj = self.metric_projection(metrics)
        metrics_proj = metrics_proj.unsqueeze(1)  # [B, 1, 384]
        metrics_encoded = self.metrics_encoder(metrics_proj)  # [B, 1, 384]

        # Traitement du code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_embed = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_embed = self.code_projection(code_embed)  # [B, seq_len, 384]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query
            code_embed,      # Key
            code_embed       # Value
        )

        # Fusion avec skip connection
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# STRATÉGIE 2: Modèle entièrement pré-entraîné (encoder et decoder pré-entraînés)
class FullPretrainedModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder PRÉ-ENTRAÎNÉ pour les métriques (adaptation de RoBERTa)
        self.metrics_encoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.metrics_encoder_config.hidden_size = Config.EMBEDDING_DIM
        self.metrics_encoder = RobertaModel(self.metrics_encoder_config)

        # Projection pour les métriques numériques vers le format attendu par transformer
        self.metrics_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),

        )

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        self.code_decoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(self.code_decoder_config)

        # Projection pour aligner les dimensions
        self.code_projection = nn.Linear(self.code_decoder_config.hidden_size, Config.EMBEDDING_DIM)

        # Couche d'attention croisée pour combiner les deux représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,

        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.ReLU()

        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Projeter les métriques et les adapter au format attendu
        metrics_embedded = self.metrics_projection(metrics).unsqueeze(1)  # [B, 1, 384]

        # Créer un masque factice pour les métriques (toutes valides)
        metrics_attention_mask = torch.ones(metrics_embedded.size(0), 1).to(metrics.device)

        # Encoder les métriques avec un transformer pré-entraîné
        metrics_encoder_output = self.metrics_encoder(
            inputs_embeds=metrics_embedded,
            attention_mask=metrics_attention_mask
        )
        metrics_encoded = metrics_encoder_output.last_hidden_state  # [B, 1, 384]

        # Encoder le code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_encoded = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_encoded = self.code_projection(code_encoded)  # [B, seq_len, 384]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query
            code_encoded,     # Key
            code_encoded      # Value
        )

        # Fusion avec skip connection
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# Fonction d'équilibrage avec BorderlineSMOTE
def apply_borderline_smote(X, y, code_lines):
    """
    Applique BorderlineSMOTE pour équilibrer les classes
    """
    print(f"Distribution avant équilibrage: {Counter(y)}")

    # Vérifier qu'il y a assez d'échantillons minoritaires
    min_samples = min(Counter(y).values())
    k_neighbors = min(3, min_samples - 1) if min_samples > 1 else 1

    try:
        smote = BorderlineSMOTE(random_state=42, k_neighbors=k_neighbors, kind='borderline-1')
        X_resampled, y_resampled = smote.fit_resample(X, y)
        print("✓ BorderlineSMOTE appliqué avec succès")
    except Exception as e:
        print(f"Erreur BorderlineSMOTE: {e}, utilisation de SMOTE standard")
        smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
        X_resampled, y_resampled = smote.fit_resample(X, y)

    print(f"Distribution après équilibrage: {Counter(y_resampled)}")

    # Adapter la liste de code
    n_original = len(code_lines)
    n_resampled = len(X_resampled)

    if n_resampled > n_original:
        # Mapping pour les indices répliqués
        indices_map = []
        for i in range(n_resampled):
            if i < n_original:
                indices_map.append(i)
            else:
                # Pour les nouveaux échantillons, choisir aléatoirement
                indices_map.append(np.random.randint(0, n_original))

        code_lines_resampled = [code_lines[idx] for idx in indices_map]
    else:
        code_lines_resampled = code_lines[:n_resampled]

    return X_resampled, y_resampled, code_lines_resampled

# Calcul des poids de classe
def calculate_class_weights(labels):
    """Calcule les poids de classe pour équilibrer la loss"""
    class_counts = Counter(labels)
    total_samples = len(labels)
    n_classes = len(class_counts)

    weights = {}
    for class_id, count in class_counts.items():
        weights[class_id] = total_samples / (n_classes * count)

    # Convertir en tensor PyTorch
    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])
    return weight_tensor

# Prétraitement des métriques
def preprocess_metrics(df):
    """Préprocessing robuste des métriques"""
    if 'name' in df.columns:
        df = df.drop(columns=['name'])

    df['bug'] = (df['bug'] > 0).astype(int)
    features = df.drop(columns=['bug'])
    labels = df['bug'].values

    # Traiter les valeurs manquantes
    if features.isnull().any().any():
        print("Valeurs manquantes détectées, remplacement par la médiane")
        features = features.fillna(features.median())

    # Supprimer les features avec variance nulle
    var_threshold = 0.01
    low_var_cols = features.var() < var_threshold
    if low_var_cols.any():
        print(f"Suppression de {low_var_cols.sum()} colonnes avec faible variance")
        features = features.loc[:, ~low_var_cols]

    # Normalisation
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    print(f"Distribution des classes: {Counter(labels)}")

    return features_scaled, labels, scaler

# Chargement des métriques et du code
def load_code_and_metrics(metrics_path, code_path):
    """Charge les métriques et le code source"""
    df_metrics = pd.read_csv(metrics_path)
    code_lines = []

    if os.path.exists(code_path):
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        with open(os.path.join(root, file), 'r', encoding='utf-8', errors='ignore') as f:
                            code = f.read()
                            code = code.strip()
                            if len(code) > 0:
                                code_lines.append(code)
                    except Exception as e:
                        print(f"Erreur lecture fichier {file}: {e}")
                        continue

    # S'assurer qu'on a au moins autant de fichiers code que de métriques
    if len(code_lines) < len(df_metrics):
        print(f"Attention: {len(code_lines)} fichiers code vs {len(df_metrics)} métriques")
        while len(code_lines) < len(df_metrics):
            if len(code_lines) > 0:
                code_lines.append(code_lines[len(code_lines) % len(code_lines)])
            else:
                code_lines.append("// Empty code file")

    return df_metrics, code_lines

# Préparation du dataset avec BorderlineSMOTE
def prepare_single_dataset(metrics_path, code_path, tokenizer, apply_balancing=True):
    """Prépare un dataset - VERSION CORRIGÉE"""
    df_metrics, code_lines = load_code_and_metrics(metrics_path, code_path)
    features, labels, scaler = preprocess_metrics(df_metrics)

    # Assurer la cohérence des tailles
    min_len = min(len(features), len(code_lines), len(labels))
    features = features[:min_len]
    code_lines = code_lines[:min_len]
    labels = labels[:min_len]

    # Appliquer l'équilibrage seulement si demandé
    if apply_balancing and len(set(labels)) > 1:
        try:
            features, labels, code_lines = apply_borderline_smote(features, labels, code_lines)
        except Exception as e:
            print(f"Erreur lors de l'équilibrage: {e}")
            print("Utilisation des données originales")

    # Tokenisation du code
    try:
        code_tokens = tokenizer(
            code_lines,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512
        )
    except Exception as e:
        print(f"Erreur lors de la tokenisation: {e}")
        code_tokens = {
            'input_ids': torch.zeros((len(features), 512), dtype=torch.long),
            'attention_mask': torch.zeros((len(features), 512), dtype=torch.long)
        }

    return features, code_tokens, labels, scaler

# Préparation de datasets multiples
def prepare_multiple_datasets(metrics_paths, code_paths, tokenizer, apply_balancing=True):
    """Prépare et combine plusieurs datasets - VERSION CORRIGÉE"""
    all_features, all_labels, all_code_lines = [], [], []

    # ÉTAPE 1: Charger et combiner TOUTES les données SANS équilibrage
    for metrics_path, code_path in zip(metrics_paths, code_paths):
        print(f"Chargement de {metrics_path}...")

        # Charger sans équilibrage individuel
        df_metrics, code_lines = load_code_and_metrics(metrics_path, code_path)
        features, labels, scaler = preprocess_metrics(df_metrics)

        # Assurer la cohérence des tailles
        min_len = min(len(features), len(code_lines), len(labels))
        features = features[:min_len]
        code_lines = code_lines[:min_len]
        labels = labels[:min_len]

        print(f"  Dataset {metrics_path}: {Counter(labels)}")

        all_features.append(features)
        all_labels.extend(labels)
        all_code_lines.extend(code_lines)

    # ÉTAPE 2: Concaténer toutes les données
    X_combined = np.vstack(all_features)
    y_combined = np.array(all_labels)
    code_combined = all_code_lines

    print(f"\nDonnées combinées - Distribution avant équilibrage: {Counter(y_combined)}")

    # ÉTAPE 3: Appliquer BorderlineSMOTE UNE SEULE FOIS sur l'ensemble combiné
    if apply_balancing and len(set(y_combined)) > 1:
        try:
            X_balanced, y_balanced, code_balanced = apply_borderline_smote(
                X_combined, y_combined, code_combined
            )
            print(f"Données finales - Distribution après équilibrage: {Counter(y_balanced)}")
        except Exception as e:
            print(f"Erreur lors de l'équilibrage: {e}")
            print("Utilisation des données originales")
            X_balanced, y_balanced, code_balanced = X_combined, y_combined, code_combined
    else:
        X_balanced, y_balanced, code_balanced = X_combined, y_combined, code_combined

    # ÉTAPE 4: Tokeniser le code source équilibré
    try:
        print("Tokenisation du code source...")
        combined_code_tokens = tokenizer(
            code_balanced,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512
        )
        print(f"✓ Tokenisation réussie: {len(code_balanced)} échantillons")
    except Exception as e:
        print(f"Erreur lors de la tokenisation: {e}")
        combined_code_tokens = {
            'input_ids': torch.zeros((len(X_balanced), 512), dtype=torch.long),
            'attention_mask': torch.zeros((len(X_balanced), 512), dtype=torch.long)
        }

    return X_balanced, combined_code_tokens, y_balanced

# Évaluation
def evaluate(model, dataloader, criterion=None):
    """Évaluation robuste du modèle"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for metrics, code_tokens, labels in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculer les métriques
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    try:
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    except:
        f1 = 0.0

    accuracy = accuracy_score(all_labels, all_preds)

    # Calcul robuste de sensibilité et spécificité
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)

        if sens > 0 and spec > 0:
            g_mean = gmean([sens, spec])
        else:
            g_mean = 0.0
    else:
        sens = spec = g_mean = 0.0

    avg_loss = total_loss / len(all_labels) if criterion is not None and len(all_labels) > 0 else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# Fonction d'entraînement principale
def train_model(model_type="hybrid"):
    """Version corrigée de train_model"""
    print(f"\n{'='*50}")
    print(f"ENTRAÎNEMENT: {model_type.upper()}")
    print(f"{'='*50}")

    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Définir les chemins
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/synapse/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/synapse/"

    # Versions pour l'entraînement
    train_versions = ["1.0", "1.1"]
    metrics_train_paths = [f"{base_metrics_path}synapse-{v}.csv" for v in train_versions]
    code_train_paths = [f"{base_code_path}synapse-{v}" for v in train_versions]

    # Version pour la validation
    val_version = "1.2"
    metrics_val_path = f"{base_metrics_path}synapse-{val_version}.csv"
    code_val_path = f"{base_code_path}synapse-{val_version}"

    # ✅ CORRECTION: Préparer les datasets d'entraînement avec BorderlineSMOTE appliqué UNE FOIS
    print("Préparation des données d'entraînement avec BorderlineSMOTE...")
    print("Chargement de toutes les versions d'entraînement...")
    X_train, train_code_tokens, y_train = prepare_multiple_datasets(
        metrics_train_paths, code_train_paths, tokenizer, apply_balancing=True
    )

    # Préparer le dataset de validation (sans équilibrage)
    print("Préparation des données de validation...")
    X_val, code_val_tokens, y_val, _ = prepare_single_dataset(
        metrics_val_path, code_val_path, tokenizer, apply_balancing=False
    )

    # Créer les datasets
    train_ds = BugDataset(X_train, train_code_tokens, y_train)
    val_ds = BugDataset(X_val, code_val_tokens, y_val)

    # Créer les DataLoaders
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Calculer les poids de classe
    class_weights = calculate_class_weights(y_train).to(Config.DEVICE)
    print(f"Poids des classes: {class_weights}")

    # Initialiser le modèle selon le type
    if model_type == "full_pretrained":
        model = FullPretrainedModel().to(Config.DEVICE)
        print("✓ Modèle avec ENCODER et DECODER pré-entraînés")
    else:
        model = HybridTransBugModel().to(Config.DEVICE)
        print("✓ Modèle avec ENCODER from scratch et DECODER pré-entraîné")

    # Optimiseur avec learning rates différenciés
    if model_type == "full_pretrained":
        # Pour le modèle entièrement pré-entraîné, utiliser des LR plus faibles pour les parties pré-entraînées
        optimizer = torch.optim.AdamW([
            {'params': model.metrics_encoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters()
                       if 'metrics_encoder' not in n and 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)
    else:
        # Pour le modèle hybride, seul le decoder est pré-entraîné
        optimizer = torch.optim.AdamW([
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters() if 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)

    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=True
    )

    # Loss avec pondération et label smoothing
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=Config.LABEL_SMOOTHING)

    # Listes pour le monitoring
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    f1_scores, gmean_scores = [], []

    best_f1 = 0
    patience_counter = 0

    for epoch in range(Config.EPOCHS):
        # Entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for batch_idx, (metrics, code_tokens, labels) in enumerate(train_loader):
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)

            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Métriques d'entraînement
        train_loss /= len(train_preds)
        train_acc = accuracy_score(train_labels_list, train_preds)
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        # Scheduler step
        scheduler.step(f1)

        print(f"Epoch {epoch+1}/{Config.EPOCHS}:")
        print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
        print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")
        print(f"  Val: Sensitivité={sens:.4f}, Spécificité={spec:.4f}")

        # Sauvegarde du meilleur modèle
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}best_model_{model_type}.pt")
            joblib.dump(cm, f"{Config.SAVE_PATH}conf_matrix_{model_type}.pkl")
            joblib.dump(val_preds, f"{Config.SAVE_PATH}val_preds_{model_type}.pkl")
            joblib.dump(val_labels, f"{Config.SAVE_PATH}val_labels_{model_type}.pkl")
            print(f"  *** Nouveau meilleur F1: {f1:.4f} ***")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= 17:
            print("Early stopping déclenché")
            break

    print(f"\n{model_type.upper()} - Meilleur F1 score: {best_f1:.4f}")

    # Visualisations
    plt.figure(figsize=(15, 5))

    # Courbe 1: Loss
    plt.subplot(1, 3, 1)
    epochs_range = range(1, len(train_losses) + 1)
    plt.plot(epochs_range, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs_range, val_losses, 'r-', label='Val Loss')
    plt.title(f'{model_type.capitalize()} - Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Courbe 2: Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, train_accuracies, 'b-', label='Train Accuracy')
    plt.plot(epochs_range, val_accuracies, 'r-', label='Val Accuracy')
    plt.title(f'{model_type.capitalize()} - Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Courbe 3: F1-score et G-mean
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, f1_scores, 'g-', label='F1-Score')
    plt.plot(epochs_range, gmean_scores, 'm-', label='G-Mean')
    plt.title(f'{model_type.capitalize()} - F1-Score and G-Mean')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"{Config.SAVE_PATH}training_curves_{model_type}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Matrice de confusion finale
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Bug', 'Bug'],
                yticklabels=['Non-Bug', 'Bug'])
    plt.title(f'{model_type.capitalize()} - Confusion Matrix (Best Model)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f"{Config.SAVE_PATH}confusion_matrix_{model_type}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Sauvegarder les métriques
    results = {
        'model_type': model_type,
        'best_f1': best_f1,
        'final_accuracy': val_accuracies[-1] if val_accuracies else 0,
        'final_sensitivity': sens,
        'final_specificity': spec,
        'final_gmean': gmean,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'f1_scores': f1_scores,
        'gmean_scores': gmean_scores
    }

    joblib.dump(results, f"{Config.SAVE_PATH}training_results_{model_type}.pkl")

    return model, results

def compare_models():
    """Compare les deux stratégies de modèles"""
    print("\n" + "="*60)
    print("COMPARAISON DES DEUX STRATÉGIES")
    print("="*60)

    # Entraîner le modèle hybride
    print("\n1. STRATÉGIE HYBRIDE (Encoder from scratch + Decoder pré-entraîné)")
    model_hybrid, results_hybrid = train_model("hybrid")

    # Entraîner le modèle entièrement pré-entraîné
    print("\n2. STRATÉGIE ENTIÈREMENT PRÉ-ENTRAÎNÉE (Encoder et Decoder pré-entraînés)")
    model_full, results_full = train_model("full_pretrained")

    # Comparaison des résultats
    print("\n" + "="*60)
    print("RÉSULTATS COMPARATIFS")
    print("="*60)

    print(f"{'Métrique':<20} {'Hybride':<15} {'Full Pretrained':<15} {'Meilleur':<15}")
    print("-" * 70)

    metrics_comparison = [
        ('F1-Score', results_hybrid['best_f1'], results_full['best_f1']),
        ('Accuracy', results_hybrid['final_accuracy'], results_full['final_accuracy']),
        ('Sensitivity', results_hybrid['final_sensitivity'], results_full['final_sensitivity']),
        ('Specificity', results_hybrid['final_specificity'], results_full['final_specificity']),
        ('G-Mean', results_hybrid['final_gmean'], results_full['final_gmean'])
    ]

    for metric_name, hybrid_score, full_score in metrics_comparison:
        better = "Hybride" if hybrid_score > full_score else "Full Pretrained" if full_score > hybrid_score else "Égalité"
        print(f"{metric_name:<20} {hybrid_score:<15.4f} {full_score:<15.4f} {better:<15}")

    # Graphique comparatif
    plt.figure(figsize=(12, 8))

    # Comparaison F1-Score au fil des époques
    plt.subplot(2, 2, 1)
    epochs_hybrid = range(1, len(results_hybrid['f1_scores']) + 1)
    epochs_full = range(1, len(results_full['f1_scores']) + 1)
    plt.plot(epochs_hybrid, results_hybrid['f1_scores'], 'b-', label='Hybride', linewidth=2)
    plt.plot(epochs_full, results_full['f1_scores'], 'r-', label='Full Pretrained', linewidth=2)
    plt.title('Comparaison F1-Score')
    plt.xlabel('Epochs')
    plt.ylabel('F1-Score')
    plt.legend()
    plt.grid(True)

    # Comparaison G-Mean
    plt.subplot(2, 2, 2)
    plt.plot(epochs_hybrid, results_hybrid['gmean_scores'], 'b-', label='Hybride', linewidth=2)
    plt.plot(epochs_full, results_full['gmean_scores'], 'r-', label='Full Pretrained', linewidth=2)
    plt.title('Comparaison G-Mean')
    plt.xlabel('Epochs')
    plt.ylabel('G-Mean')
    plt.legend()
    plt.grid(True)

    # Comparaison Accuracy
    plt.subplot(2, 2, 3)
    plt.plot(epochs_hybrid, results_hybrid['val_accuracies'], 'b-', label='Hybride', linewidth=2)
    plt.plot(epochs_full, results_full['val_accuracies'], 'r-', label='Full Pretrained', linewidth=2)
    plt.title('Comparaison Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Graphique en barres des métriques finales
    plt.subplot(2, 2, 4)
    metrics_names = ['F1-Score', 'Accuracy', 'Sensitivity', 'Specificity', 'G-Mean']
    hybrid_values = [results_hybrid['best_f1'], results_hybrid['final_accuracy'],
                    results_hybrid['final_sensitivity'], results_hybrid['final_specificity'],
                    results_hybrid['final_gmean']]
    full_values = [results_full['best_f1'], results_full['final_accuracy'],
                  results_full['final_sensitivity'], results_full['final_specificity'],
                  results_full['final_gmean']]

    x = np.arange(len(metrics_names))
    width = 0.35

    plt.bar(x - width/2, hybrid_values, width, label='Hybride', alpha=0.8)
    plt.bar(x + width/2, full_values, width, label='Full Pretrained', alpha=0.8)

    plt.title('Comparaison des Métriques Finales')
    plt.xlabel('Métriques')
    plt.ylabel('Score')
    plt.xticks(x, metrics_names, rotation=45)
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{Config.SAVE_PATH}models_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Recommandation
    print("\n" + "="*60)
    print("RECOMMANDATION")
    print("="*60)

    if results_hybrid['best_f1'] > results_full['best_f1']:
        best_model = "Hybride"
        best_results = results_hybrid
    else:
        best_model = "Full Pretrained"
        best_results = results_full

    print(f"✓ Le modèle {best_model} obtient les meilleurs résultats")
    print(f"  - F1-Score: {best_results['best_f1']:.4f}")
    print(f"  - G-Mean: {best_results['final_gmean']:.4f}")
    print(f"  - Accuracy: {best_results['final_accuracy']:.4f}")

    # Analyse des avantages
    print(f"\nAnalyse des stratégies:")
    print("• Modèle Hybride:")
    print("  - Encoder from scratch pour les métriques")
    print("  - Decoder pré-entraîné pour le code source")
    print("  - Plus flexible pour l'adaptation aux métriques spécifiques")

    print("• Modèle Full Pretrained:")
    print("  - Encoder et Decoder pré-entraînés")
    print("  - Tire parti de plus de connaissances pré-entraînées")
    print("  - Peut nécessiter moins d'entraînement")

    return best_model, best_results

def main():
    """Fonction principale pour lancer la comparaison"""
    print("🚀 Démarrage de l'entraînement et comparaison des modèles de détection de bugs")
    print("Utilisation de BorderlineSMOTE pour l'équilibrage des classes")

    try:
        best_model, best_results = compare_models()

        print(f"\n🎉 Entraînement terminé avec succès!")
        print(f"Meilleur modèle: {best_model}")
        print(f"Résultats sauvegardés dans: {Config.SAVE_PATH}")

    except Exception as e:
        print(f"❌ Erreur lors de l'entraînement: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
# Installation des dépendances
!pip install transformers imbalanced-learn scikit-learn torch pandas numpy matplotlib seaborn joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter
import re
import os
import random

# Import des bibliothèques optionnelles
try:
    from imblearn.over_sampling import BorderlineSMOTE, SMOTE
    IMBLEARN_AVAILABLE = True
except ImportError:
    print("⚠️ imblearn non disponible - SMOTE désactivé")
    IMBLEARN_AVAILABLE = False

try:
    from transformers import RobertaTokenizer, RobertaModel, RobertaConfig, AutoModel, AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    print("❌ transformers non disponible - installez-le avec: !pip install transformers")
    TRANSFORMERS_AVAILABLE = False
    raise ImportError("Bibliothèques manquantes")

# Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------
# Reproductibilité - Seeds
# ---------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =============================================================================
# CONFIGURATION POUR GOOGLE COLAB
# =============================================================================
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 20
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 768
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2

    # Chemins Google Colab
    BASE_METRICS_PATH = "/content/drive/MyDrive/bug/bug-data"
    BASE_CODE_PATH = "/content/drive/MyDrive/bug/bug-data/source-code/source-code"
    SAVE_PATH = "/content/drive/MyDrive/bug/results"

    # Paramètres pour l'équilibrage
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    # Paramètres pour l'alignement
    MAX_CODE_LENGTH = 512

    @classmethod
    def initialize_paths(cls):
        """Initialise les chemins pour Google Colab"""
        print("📍 INITIALISATION DES CHEMINS GOOGLE COLAB")

        if os.path.exists(cls.BASE_METRICS_PATH):
            print(f"✅ Chemin métriques: {cls.BASE_METRICS_PATH}")
        else:
            print(f"❌ Chemin métriques inaccessible: {cls.BASE_METRICS_PATH}")
            cls.BASE_METRICS_PATH = None

        if os.path.exists(cls.BASE_CODE_PATH):
            print(f"✅ Chemin code source: {cls.BASE_CODE_PATH}")
        else:
            print(f"❌ Chemin code source inaccessible: {cls.BASE_CODE_PATH}")
            cls.BASE_CODE_PATH = None

        os.makedirs(cls.SAVE_PATH, exist_ok=True)
        print(f"✅ Dossier sauvegarde: {cls.SAVE_PATH}")

# Configuration des projets (sera remplie automatiquement)
PROJECTS_CONFIG = {}

# =============================================================================
# FONCTIONS POUR TROUVER LES FICHIERS
# =============================================================================
def find_csv_file(project, version):
    """Trouve le fichier CSV avec différentes conventions de nommage"""
    possible_patterns = [
        f"{project}-{version}.csv",
        f"{project}{version}.csv",
        f"{project}_{version}.csv",
        f"{project}-{version.replace('.', '_')}.csv",
        f"{project}{version.replace('.', '')}.csv",
        f"{project}.csv"
    ]

    project_dir = os.path.join(Config.BASE_METRICS_PATH, project)
    if os.path.exists(project_dir):
        for pattern in possible_patterns:
            file_path = os.path.join(project_dir, pattern)
            if os.path.exists(file_path):
                return file_path

    for pattern in possible_patterns:
        file_path = os.path.join(Config.BASE_METRICS_PATH, pattern)
        if os.path.exists(file_path):
            return file_path

    search_dirs = [Config.BASE_METRICS_PATH]
    if os.path.exists(os.path.join(Config.BASE_METRICS_PATH, project)):
        search_dirs.append(os.path.join(Config.BASE_METRICS_PATH, project))

    for search_dir in search_dirs:
        if os.path.exists(search_dir):
            for file in os.listdir(search_dir):
                if file.endswith('.csv') and project in file.lower():
                    return os.path.join(search_dir, file)

    return None

def find_code_directory(project, version):
    """Trouve le dossier de code avec différentes conventions"""
    # EXCLURE ivy car pas de code source
    if project.lower() == 'ivy':
        print(f"⚠️  Projet ivy exclu - pas de code source")
        return None

    possible_patterns = [
        f"{project}-{version}",
        f"{project}{version}",
        f"{project}_{version}",
        f"{project}-{version.replace('.', '_')}",
        f"{project}{version.replace('.', '')}",
        f"{project}",
        f"{project}-source",
        f"{project}-code"
    ]

    project_parent_dir = os.path.join(Config.BASE_CODE_PATH, project)
    if os.path.exists(project_parent_dir):
        for pattern in possible_patterns:
            dir_path = os.path.join(project_parent_dir, pattern)
            if os.path.exists(dir_path):
                return dir_path

    for pattern in possible_patterns:
        dir_path = os.path.join(Config.BASE_CODE_PATH, pattern)
        if os.path.exists(dir_path):
            return dir_path

    if os.path.exists(Config.BASE_CODE_PATH):
        for item in os.listdir(Config.BASE_CODE_PATH):
            item_path = os.path.join(Config.BASE_CODE_PATH, item)
            if os.path.isdir(item_path) and project in item.lower():
                return item_path

    return None

# =============================================================================
# EXPLORATION DE LA STRUCTURE RÉELLE
# =============================================================================
def explore_real_directory_structure():
    """Explore la structure réelle des dossiers"""
    print("\n🔍 EXPLORATION DE LA STRUCTURE RÉELLE DES DOSSIERS")
    print("=" * 80)

    if Config.BASE_METRICS_PATH and os.path.exists(Config.BASE_METRICS_PATH):
        print(f"📁 Dossier métriques: {Config.BASE_METRICS_PATH}")
        items = os.listdir(Config.BASE_METRICS_PATH)
        print(f"   Contenu ({len(items)} éléments):")

        for item in items[:15]:
            item_path = os.path.join(Config.BASE_METRICS_PATH, item)
            if os.path.isdir(item_path):
                csv_files = [f for f in os.listdir(item_path) if f.endswith('.csv')]
                print(f"   📂 {item}/ - {len(csv_files)} fichiers CSV")
            elif item.endswith('.csv'):
                print(f"   📄 {item}")
    else:
        print("❌ Dossier métriques non accessible")

    if Config.BASE_CODE_PATH and os.path.exists(Config.BASE_CODE_PATH):
        print(f"\n📁 Dossier code source: {Config.BASE_CODE_PATH}")
        items = os.listdir(Config.BASE_CODE_PATH)
        print(f"   Contenu ({len(items)} éléments):")

        for item in items[:10]:
            item_path = os.path.join(Config.BASE_CODE_PATH, item)
            if os.path.isdir(item_path):
                print(f"   📂 {item}/")
    else:
        print("❌ Dossier code source non accessible")

    print("=" * 80)

# =============================================================================
# DÉCOUVERTE AUTOMATIQUE DE TOUS LES PROJETS (EXCLURE ivy)
# =============================================================================
def discover_all_projects_and_versions():
    """Découvre automatiquement TOUS les projets et leurs versions (exclut ivy)"""
    print("\n🔍 DÉCOUVERTE AUTOMATIQUE DE TOUS LES PROJETS ET VERSIONS")
    print("⚠️  Exclusion du projet 'ivy' (pas de code source)")

    projects_config = {}

    if not Config.BASE_METRICS_PATH or not os.path.exists(Config.BASE_METRICS_PATH):
        print("❌ Dossier métriques non accessible")
        return projects_config

    # Liste de tous les projets connus dans le dataset PROMISE (sans ivy)
    all_known_projects = [
        'ant', 'camel', 'jedit', 'log4j', 'lucene', 'poi', 'synapse', 'velocity', 'xalan', 'xerces'
    ]

    # PROJETS À EXCLURE (pas de code source)
    EXCLUDED_PROJECTS = ['ivy']

    for item in os.listdir(Config.BASE_METRICS_PATH):
        item_path = os.path.join(Config.BASE_METRICS_PATH, item)

        if os.path.isdir(item_path):
            project_name = item.lower()

            # EXCLURE ivy et autres projets sans code source
            if project_name in EXCLUDED_PROJECTS:
                print(f"🚫 Projet exclu: {project_name} (pas de code source)")
                continue

            # Inclure TOUS les autres projets
            if project_name not in all_known_projects:
                # Vérifier si c'est un projet valide quand même
                csv_files = [f for f in os.listdir(item_path) if f.endswith('.csv')]
                if len(csv_files) > 0:
                    print(f"⚠️  Projet inconnu mais avec données: {project_name}")
                    all_known_projects.append(project_name)
                else:
                    continue

            versions = []

            for file in os.listdir(item_path):
                if file.endswith('.csv'):
                    file_lower = file.lower()
                    if file_lower.startswith(project_name):
                        version_part = file_lower.replace(project_name, '').replace('.csv', '')
                        version_part = version_part.strip('-_')
                        if version_part:
                            versions.append(version_part)
                    else:
                        versions.append(file.replace('.csv', ''))

            if versions:
                projects_config[project_name] = sorted(versions)
                print(f"✅ {project_name}: {len(versions)} versions - {versions}")

        elif item.endswith('.csv'):
            # Fichier CSV directement à la racine
            project_name = item.replace('.csv', '').split('-')[0].split('_')[0].lower()

            # EXCLURE ivy
            if project_name in EXCLUDED_PROJECTS:
                print(f"🚫 Fichier exclu: {item} (projet {project_name} sans code source)")
                continue

            if project_name not in all_known_projects:
                # Vérifier si c'est un projet connu
                if any(known_proj in item.lower() for known_proj in all_known_projects):
                    for known_proj in all_known_projects:
                        if known_proj in item.lower():
                            project_name = known_proj
                            break

            version = item.replace('.csv', '').replace(project_name, '').strip('-_')
            if not version:
                version = 'default'

            if project_name not in projects_config:
                projects_config[project_name] = []
            projects_config[project_name].append(version)
            print(f"✅ {project_name}: version {version} (fichier racine)")

    print(f"\n📊 Total: {len(projects_config)} projets découverts (ivy exclu)")
    print(f"📋 Liste complète: {list(projects_config.keys())}")
    return projects_config

# =============================================================================
# FONCTIONS D'ALIGNEMENT ET CHARGEMENT DES DONNÉES
# =============================================================================
def extract_filename_from_path(file_path):
    if pd.isna(file_path) or file_path is None:
        return []
    base_name = os.path.basename(str(file_path))
    name_without_ext = os.path.splitext(base_name)[0]
    cleaned_name = re.sub(r'[^a-zA-Z0-9]', '', name_without_ext.lower())
    if not cleaned_name:
        return []
    variations = set()
    variations.add(cleaned_name)
    variations.add(cleaned_name.replace('test', '').replace('testcase', ''))
    variations.add(re.sub(r'\d+', '', cleaned_name))
    return [v for v in variations if v and len(v) > 1]

def align_metrics_with_code(df_metrics, code_files_dict):
    aligned_metrics = []
    aligned_code = []
    aligned_labels = []
    aligned_filenames = []

    code_index = {}
    for code_filename, code_content in code_files_dict.items():
        code_variations = extract_filename_from_path(code_filename)
        for variation in code_variations:
            if variation:
                code_index[variation] = (code_content, code_filename)

    numeric_cols_global = [c for c in df_metrics.columns if c not in ['name', 'bug'] and pd.api.types.is_numeric_dtype(df_metrics[c])]

    for idx, row in df_metrics.iterrows():
        if 'name' in row:
            metric_variations = extract_filename_from_path(str(row['name']))
        else:
            metric_variations = [f"file{idx}"]

        matching_code = None
        for metric_var in metric_variations:
            if metric_var in code_index:
                matching_code, matched_filename = code_index[metric_var]
                break

        if matching_code is None and code_files_dict:
            matching_code, matched_filename = next(iter(code_index.values()))

        if matching_code is not None:
            metric_data = {}
            for col in numeric_cols_global:
                try:
                    value = row.get(col, np.nan)
                    metric_data[col] = float(value) if pd.notna(value) else 0.0
                except Exception:
                    metric_data[col] = 0.0

            aligned_metrics.append(metric_data)
            aligned_code.append(matching_code)
            aligned_labels.append(1 if row.get('bug', 0) > 0 else 0)
            aligned_filenames.append(matched_filename)

    print(f"✓ {len(aligned_metrics)} échantillons alignés")
    return aligned_metrics, aligned_code, aligned_labels, aligned_filenames

def load_and_align_data(metrics_path, code_path):
    try:
        df_metrics = pd.read_csv(metrics_path)
        print(f"✓ {len(df_metrics)} métriques chargées")

        if 'bug' in df_metrics.columns:
            bug_counts = df_metrics['bug'].value_counts()
            print(f"   Distribution des bugs: {bug_counts.to_dict()}")

    except Exception as e:
        print(f"✗ Erreur chargement métriques: {e}")
        return [], [], [], []

    code_files_dict = {}
    if os.path.exists(code_path):
        java_files_count = 0
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        file_path = os.path.join(root, file)
                        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                            code_content = f.read().strip()
                            if code_content and len(code_content) > 10:
                                code_files_dict[file] = code_content
                                java_files_count += 1
                    except Exception:
                        continue
        print(f"✓ {java_files_count} fichiers Java chargés")
    else:
        print(f"⚠️ Dossier code non trouvé: {code_path}")
        return [], [], [], []

    if not code_files_dict:
        print("⚠️ Aucun fichier code valide trouvé!")
        return [], [], [], []

    return align_metrics_with_code(df_metrics, code_files_dict)

def preprocess_aligned_metrics(metrics_list, labels):
    if not metrics_list:
        return np.array([]), np.array([]), None

    df_metrics = pd.DataFrame(metrics_list)
    df_metrics = df_metrics.fillna(0.0)

    numeric_cols = df_metrics.select_dtypes(include=[np.number]).columns
    df_metrics = df_metrics[numeric_cols]

    var_threshold = 0.001
    variances = df_metrics.var()
    low_var_cols = variances < var_threshold
    if low_var_cols.any():
        df_metrics = df_metrics.loc[:, ~low_var_cols]
        print(f"✓ {low_var_cols.sum()} colonnes à faible variance supprimées")

    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(df_metrics)

    print(f"✓ Métriques prétraitées: {features_scaled.shape}")
    return features_scaled, np.array(labels), scaler

def prepare_aligned_dataset(metrics_path, code_path, tokenizer):
    """Prépare un dataset aligné avec tokenization cohérente"""
    metrics_list, code_lines, labels, filenames = load_and_align_data(metrics_path, code_path)

    if not metrics_list:
        print("❌ Aucune donnée alignée trouvée")
        return None, None, None, None, None

    features, labels, scaler = preprocess_aligned_metrics(metrics_list, labels)

    if len(features) == 0:
        print("❌ Aucune feature après prétraitement")
        return None, None, None, None, None

    # Tokenization cohérente
    original_code_tokens = tokenizer(
        code_lines,
        padding='max_length',
        truncation=True,
        max_length=Config.MAX_CODE_LENGTH,
        return_tensors='pt',
        return_attention_mask=True
    )

    print(f"✅ Dataset préparé: {len(features)} échantillons")
    return features, original_code_tokens, labels, scaler, filenames

# =============================================================================
# ARCHITECTURES DES MODÈLES
# =============================================================================
class TransformerEncoderFromScratch(nn.Module):
    """Encoder Transformer from scratch pour les métriques"""
    def __init__(self, input_dim, embedding_dim, n_heads, num_layers, dropout):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, embedding_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=n_heads,
            dim_feedforward=embedding_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        x = self.input_projection(x).unsqueeze(1)
        encoded = self.transformer_encoder(x)
        return encoded.squeeze(1)

class PretrainedMetricsEncoder(nn.Module):
    """Encoder préentraîné complet pour les métriques"""
    def __init__(self, input_dim, embedding_dim, dropout):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, embedding_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.encoder(x)

class HybridBugPredictorV1(nn.Module):
    """Version 1: Encoder Transformer from scratch (métriques) + Decoder pretrained (code)"""
    def __init__(self, config, metrics_input_dim):
        super().__init__()
        self.config = config

        self.metrics_encoder = TransformerEncoderFromScratch(
            input_dim=metrics_input_dim,
            embedding_dim=config.EMBEDDING_DIM,
            n_heads=config.N_HEADS,
            num_layers=config.NUM_LAYERS,
            dropout=config.DROPOUT
        )

        self.code_encoder = RobertaModel.from_pretrained("microsoft/codebert-base")
        for param in list(self.code_encoder.parameters())[:-4]:
            param.requires_grad = True

        self.classifier = nn.Sequential(
            nn.Linear(config.EMBEDDING_DIM * 2, 256),
            nn.ReLU(),
            nn.Dropout(config.DROPOUT),
            nn.Linear(256, config.NUM_CLASSES)
        )

    def forward(self, metrics_features, code_input_ids, code_attention_mask):
        metrics_embeddings = self.metrics_encoder(metrics_features)
        code_outputs = self.code_encoder(input_ids=code_input_ids, attention_mask=code_attention_mask)
        code_embeddings = code_outputs.last_hidden_state[:, 0, :]

        fused_embeddings = torch.cat([metrics_embeddings, code_embeddings], dim=1)
        logits = self.classifier(fused_embeddings)
        return logits

class HybridBugPredictorV2(nn.Module):
    """Version 2: Encoder pretrained complet (métriques) + Decoder pretrained (code)"""
    def __init__(self, config, metrics_input_dim):
        super().__init__()
        self.config = config

        self.metrics_encoder = PretrainedMetricsEncoder(
            input_dim=metrics_input_dim,
            embedding_dim=config.EMBEDDING_DIM,
            dropout=config.DROPOUT
        )

        self.code_encoder = RobertaModel.from_pretrained("microsoft/codebert-base")
        for param in list(self.code_encoder.parameters())[:-4]:
            param.requires_grad = True

        self.classifier = nn.Sequential(
            nn.Linear(config.EMBEDDING_DIM * 2, 256),
            nn.ReLU(),
            nn.Dropout(config.DROPOUT),
            nn.Linear(256, config.NUM_CLASSES)
        )

    def forward(self, metrics_features, code_input_ids, code_attention_mask):
        metrics_embeddings = self.metrics_encoder(metrics_features)
        code_outputs = self.code_encoder(input_ids=code_input_ids, attention_mask=code_attention_mask)
        code_embeddings = code_outputs.last_hidden_state[:, 0, :]

        fused_embeddings = torch.cat([metrics_embeddings, code_embeddings], dim=1)
        logits = self.classifier(fused_embeddings)
        return logits

# =============================================================================
# DATASET PERSONNALISÉ
# =============================================================================
class BugDataset(Dataset):
    def __init__(self, metrics_features, code_tokens, labels):
        self.metrics_features = torch.FloatTensor(metrics_features) if metrics_features is not None else None
        self.code_input_ids = code_tokens['input_ids'] if code_tokens is not None else None
        self.code_attention_mask = code_tokens['attention_mask'] if code_tokens is not None else None
        self.labels = torch.LongTensor(labels) if labels is not None else None

    def __len__(self):
        return len(self.labels) if self.labels is not None else 0

    def __getitem__(self, idx):
        return {
            'metrics': self.metrics_features[idx],
            'code_input_ids': self.code_input_ids[idx],
            'code_attention_mask': self.code_attention_mask[idx],
            'labels': self.labels[idx]
        }

# =============================================================================
# FONCTIONS D'ÉVALUATION ROBUSTES
# =============================================================================
def robust_gmean(y_true, y_pred):
    """Calcul robuste du G-Mean"""
    try:
        cm = confusion_matrix(y_true, y_pred)
        if cm.shape[0] < 2:
            return 0.0

        TN, FP, FN, TP = cm.ravel()

        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0

        if specificity > 0 and sensitivity > 0:
            gmean_value = np.sqrt(specificity * sensitivity)
        else:
            gmean_value = 0.0

        if 0 < gmean_value < 0.0001:
            return 0.0001
        else:
            return round(gmean_value, 4)

    except Exception as e:
        print(f"⚠️ Erreur dans le calcul du G-Mean: {e}")
        return 0.0

def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            metrics = batch['metrics'].to(device)
            code_input_ids = batch['code_input_ids'].to(device)
            code_attention_mask = batch['code_attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(metrics, code_input_ids, code_attention_mask)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    if len(all_preds) == 0:
        return 0.0, 0.0, 0.0, [], []

    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    gmean_val = robust_gmean(all_labels, all_preds)

    return accuracy, f1, gmean_val, all_preds, all_labels

# =============================================================================
# STRATÉGIE CROSS-PROJECT CORRECTE AVEC TOUS LES PROJETS (EXCLURE ivy)
# =============================================================================
def prepare_cross_project_data(target_project):
    """Prépare les données selon la stratégie exacte demandée"""

    print(f"\n🎯 PRÉPARATION DES DONNÉES POUR {target_project}")
    print("=" * 60)

    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Stockage des données d'entraînement
    all_train_features = []
    all_train_labels = []
    all_train_input_ids = []
    all_train_attention_masks = []

    # 1. TRAIN: (n-1) versions du projet cible
    if target_project in PROJECTS_CONFIG:
        versions = PROJECTS_CONFIG[target_project]
        if len(versions) > 1:
            train_versions = versions[:-1]  # n-1 premières versions
            print(f"📥 TRAIN: {len(train_versions)} versions de {target_project}: {train_versions}")

            for version in train_versions:
                metrics_path = find_csv_file(target_project, version)
                code_path = find_code_directory(target_project, version)

                if metrics_path and code_path:
                    print(f"  📊 Chargement {target_project}-{version}")
                    data = prepare_aligned_dataset(metrics_path, code_path, tokenizer)

                    if data[0] is not None and len(data[0]) > 0:
                        features, tokens, labels, _, _ = data
                        all_train_features.append(features)
                        all_train_labels.extend(labels)
                        all_train_input_ids.append(tokens['input_ids'])
                        all_train_attention_masks.append(tokens['attention_mask'])
                        print(f"  ✅ Ajouté: {len(features)} échantillons")

    # 2. TRAIN: TOUTES les versions des autres projets (ivy exclu automatiquement)
    other_projects = [p for p in PROJECTS_CONFIG.keys() if p != target_project]
    print(f"📥 TRAIN: Toutes les versions des {len(other_projects)} autres projets: {other_projects}")

    for other_project in other_projects:
        if other_project in PROJECTS_CONFIG:
            versions = PROJECTS_CONFIG[other_project]
            print(f"  📥 Projet {other_project}: {len(versions)} versions")

            for version in versions:
                metrics_path = find_csv_file(other_project, version)
                code_path = find_code_directory(other_project, version)

                if metrics_path and code_path:
                    print(f"    📊 Chargement {other_project}-{version}")
                    data = prepare_aligned_dataset(metrics_path, code_path, tokenizer)

                    if data[0] is not None and len(data[0]) > 0:
                        features, tokens, labels, _, _ = data
                        all_train_features.append(features)
                        all_train_labels.extend(labels)
                        all_train_input_ids.append(tokens['input_ids'])
                        all_train_attention_masks.append(tokens['attention_mask'])
                        print(f"    ✅ Ajouté: {len(features)} échantillons")
                elif metrics_path and not code_path:
                    print(f"    ⚠️  Métriques trouvées mais code non disponible pour {other_project}-{version}")

    # 3. TEST: Dernière version (n) du projet cible
    test_features, test_tokens, test_labels = None, None, None
    if target_project in PROJECTS_CONFIG:
        versions = PROJECTS_CONFIG[target_project]
        if len(versions) > 0:
            test_version = versions[-1]  # Dernière version
            print(f"\n📤 TEST: Dernière version de {target_project}: {test_version}")

            test_metrics_path = find_csv_file(target_project, test_version)
            test_code_path = find_code_directory(target_project, test_version)

            if test_metrics_path and test_code_path:
                print(f"📊 Chargement données de TEST: {target_project}-{test_version}")
                test_data = prepare_aligned_dataset(test_metrics_path, test_code_path, tokenizer)

                if test_data[0] is not None and len(test_data[0]) > 0:
                    test_features, test_tokens, test_labels, _, _ = test_data
                    print(f"✅ Données de test: {len(test_features)} échantillons")
            else:
                print(f"❌ Données de test non disponibles pour {target_project}-{test_version}")

    # Vérification finale
    if len(all_train_features) == 0:
        print("❌ Aucune donnée d'entraînement trouvée")
        return None, None, None, None, None, None

    if test_features is None:
        print("❌ Aucune donnée de test trouvée")
        return None, None, None, None, None, None

    # Concaténation des données d'entraînement
    print(f"\n🔨 CONCATÉNATION DES DONNÉES D'ENTRAÎNEMENT")

    # Concaténer les features
    train_features_combined = np.vstack(all_train_features)

    # Concaténer les tokens (tous ont la même taille grâce au padding fixe)
    train_input_ids_combined = torch.cat(all_train_input_ids, dim=0)
    train_attention_masks_combined = torch.cat(all_train_attention_masks, dim=0)

    train_tokens_combined = {
        'input_ids': train_input_ids_combined,
        'attention_mask': train_attention_masks_combined
    }

    print(f"📊 RÉSUMÉ FINAL:")
    print(f"   TRAIN: {len(train_features_combined)} échantillons")
    print(f"   TEST:  {len(test_features)} échantillons")
    print(f"   Distribution TRAIN: {Counter(all_train_labels)}")
    print(f"   Distribution TEST:  {Counter(test_labels)}")
    print(f"   Projets utilisés pour l'entraînement: {len(PROJECTS_CONFIG)} projets")

    metrics_input_dim = train_features_combined.shape[1]

    return (train_features_combined, train_tokens_combined, all_train_labels,
            test_features, test_tokens, test_labels, metrics_input_dim)

def train_and_compare_models(target_project, train_metrics, train_code_tokens, train_labels,
                           test_metrics, test_code_tokens, test_labels, metrics_input_dim):
    """Entraîne et compare les deux architectures hybrides"""

    print(f"\n🔍 COMPARAISON DES ARCHITECTURES POUR {target_project}")
    print("=" * 60)

    if train_metrics is None or len(train_metrics) == 0:
        print("❌ Données d'entraînement manquantes")
        return {}

    # Créer les datasets
    train_dataset = BugDataset(train_metrics, train_code_tokens, train_labels)
    test_dataset = BugDataset(test_metrics, test_code_tokens, test_labels)

    if len(train_dataset) == 0:
        print("❌ Dataset d'entraînement vide")
        return {}

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)

    # Initialiser les modèles
    models = {
        'Transformer_Scratch_Metrics': HybridBugPredictorV1(Config, metrics_input_dim),
        'Pretrained_Metrics': HybridBugPredictorV2(Config, metrics_input_dim)
    }

    results = {}

    for model_name, model in models.items():
        print(f"\n🚀 ENTRAÎNEMENT {model_name} POUR {target_project}")

        model = model.to(Config.DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
        criterion = nn.CrossEntropyLoss()

        best_gmean = 0.0
        best_model_state = None

        for epoch in range(Config.EPOCHS):
            model.train()
            total_loss = 0
            batch_count = 0

            for batch in train_loader:
                try:
                    optimizer.zero_grad()

                    metrics = batch['metrics'].to(Config.DEVICE)
                    code_input_ids = batch['code_input_ids'].to(Config.DEVICE)
                    code_attention_mask = batch['code_attention_mask'].to(Config.DEVICE)
                    labels = batch['labels'].to(Config.DEVICE)

                    outputs = model(metrics, code_input_ids, code_attention_mask)
                    loss = criterion(outputs, labels)

                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)
                    optimizer.step()

                    total_loss += loss.item()
                    batch_count += 1
                except Exception as e:
                    print(f"❌ Erreur pendant l'entraînement: {e}")
                    continue

            if batch_count == 0:
                continue

            # Évaluation
            accuracy, f1, gmean_val, _, _ = evaluate_model(model, test_loader, Config.DEVICE)

            avg_loss = total_loss / batch_count
            print(f"Epoch {epoch+1}/{Config.EPOCHS} | Loss: {avg_loss:.4f} | "
                  f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | G-Mean: {gmean_val:.4f}")

            if gmean_val > best_gmean:
                best_gmean = gmean_val
                best_model_state = model.state_dict().copy()

        # Charger le meilleur modèle
        if best_model_state is not None:
            model.load_state_dict(best_model_state)

        accuracy, f1, gmean_val, preds, true_labels = evaluate_model(model, test_loader, Config.DEVICE)

        results[model_name] = {
            'accuracy': accuracy,
            'f1': f1,
            'gmean': gmean_val,
            'predictions': preds,
            'true_labels': true_labels
        }

        print(f"\n🎯 RÉSULTATS FINAUX {model_name}:")
        print(f"   Accuracy: {accuracy:.4f}")
        print(f"   F1-Score: {f1:.4f}")
        print(f"   G-Mean: {gmean_val:.4f}")

        # Matrice de confusion
        if len(preds) > 0:
            cm = confusion_matrix(true_labels, preds)
            plt.figure(figsize=(6, 5))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
            plt.title(f'Matrice de Confusion - {target_project} - {model_name}')
            plt.ylabel('Vrai')
            plt.xlabel('Prédit')
            plt.savefig(os.path.join(Config.SAVE_PATH, f'confusion_{target_project}_{model_name}.png'),
                       dpi=300, bbox_inches='tight')
            plt.show()

    return results

def train_cross_project(target_project):
    """Entraîne le modèle avec la stratégie cross-project CORRECTE"""
    print(f"\n🎯 DÉMARRAGE ENTRAÎNEMENT CROSS-PROJECT POUR {target_project}")
    print("=" * 80)

    if target_project not in PROJECTS_CONFIG:
        print(f"❌ Projet {target_project} non trouvé")
        return {}

    # Préparer les données selon la stratégie exacte
    data = prepare_cross_project_data(target_project)
    if data is None:
        return {}

    (train_features, train_tokens, train_labels,
     test_features, test_tokens, test_labels, metrics_input_dim) = data

    # Entraîner et comparer les modèles
    results = train_and_compare_models(
        target_project,
        train_features, train_tokens, train_labels,
        test_features, test_tokens, test_labels,
        metrics_input_dim
    )

    # Sauvegarder les résultats
    if results:
        comparison_results = []
        for model_name, metrics in results.items():
            comparison_results.append({
                'Project': target_project,
                'Model': model_name,
                'Accuracy': metrics['accuracy'],
                'F1-Score': metrics['f1'],
                'G-Mean': metrics['gmean'],
                'Train_Samples': len(train_features),
                'Test_Samples': len(test_features),
                'Total_Projects': len(PROJECTS_CONFIG)
            })

        comparison_df = pd.DataFrame(comparison_results)
        results_file = os.path.join(Config.SAVE_PATH, 'model_comparison_results.csv')

        if os.path.exists(results_file):
            existing_df = pd.read_csv(results_file)
            comparison_df = pd.concat([existing_df, comparison_df], ignore_index=True)

        comparison_df.to_csv(results_file, index=False)
        print(f"✅ Résultats sauvegardés dans {results_file}")

    return results

# =============================================================================
# FONCTION PRINCIPALE
# =============================================================================
def main():
    """Fonction principale pour Google Colab"""
    print("🚀 DÉMARRAGE AVEC STRATÉGIE CROSS-PROJECT COMPLÈTE")
    print("=" * 80)
    print("STRATÉGIE: Pour chaque projet cible:")
    print("  • TRAIN: (n-1) versions du projet + TOUTES les versions des autres projets")
    print("  • TEST:  Dernière version (n) du projet cible")
    print("⚠️  EXCLUSION: Projet 'ivy' (pas de code source)")
    print("=" * 80)

    # Initialiser les chemins
    Config.initialize_paths()

    if not Config.BASE_METRICS_PATH:
        print("❌ Impossible de trouver les données. Arrêt.")
        return

    # Explorer la structure
    explore_real_directory_structure()

    # Découvrir TOUS les projets (ivy exclu)
    global PROJECTS_CONFIG
    PROJECTS_CONFIG = discover_all_projects_and_versions()

    if not PROJECTS_CONFIG:
        print("❌ Aucun projet trouvé. Vérifiez vos données.")
        return

    print(f"\n🎯 PROJETS DISPONIBLES: {len(PROJECTS_CONFIG)} projets (ivy exclu)")
    print(f"📋 Liste: {list(PROJECTS_CONFIG.keys())}")

    # Projets cibles pour l'évaluation
    target_projects = ['camel', 'log4j', 'synapse', 'xerces']
    available_targets = [p for p in target_projects if p in PROJECTS_CONFIG]

    if not available_targets:
        print("❌ Aucun projet cible disponible")
        return

    print(f"\n🎯 PROJETS CIBLES POUR ÉVALUATION: {available_targets}")
    print(f"📊 Projets disponibles pour l'entraînement: {len(PROJECTS_CONFIG)} projets")

    all_results = {}
    for target_project in available_targets:
        print(f"\n{'='*80}")
        print(f"🎯 TRAITEMENT DU PROJET CIBLE: {target_project}")
        print(f"📚 Utilisation de {len(PROJECTS_CONFIG)} projets pour l'entraînement")
        print(f"{'='*80}")

        results = train_cross_project(target_project)
        if results:
            all_results[target_project] = results

    # Résumé final
    if all_results:
        print("\n" + "="*100)
        print("📊 RÉSUMÉ FINAL COMPARATIF - STRATÉGIE CROSS-PROJECT COMPLÈTE")
        print("="*100)

        summary_data = []
        for project, models_results in all_results.items():
            for model_name, metrics in models_results.items():
                summary_data.append({
                    'Project': project,
                    'Model': model_name,
                    'Accuracy': f"{metrics['accuracy']:.4f}",
                    'F1-Score': f"{metrics['f1']:.4f}",
                    'G-Mean': f"{metrics['gmean']:.4f}"
                })

        summary_df = pd.DataFrame(summary_data)
        print("\n" + summary_df.to_string(index=False))

        summary_file = os.path.join(Config.SAVE_PATH, 'final_comparison_summary.csv')
        summary_df.to_csv(summary_file, index=False)
        print(f"\n✅ Résumé final sauvegardé dans: {summary_file}")

if __name__ == "__main__":
    main()